## Segmentation of individual kidneys 
#### Tissue segmentation to obtain individual kidneys from the Stereo-seq slides with three kidneys

In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import PIL
PIL.Image.MAX_IMAGE_PIXELS = None
import numpy as np
from numpy import asarray
import seaborn as sns
import os
import anndata as ad
import warnings
import pickle as pkl
from matplotlib.backends.backend_pdf import PdfPages

from collections import Counter

warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
datadir = '/exports/archive/hg-funcgenom-research/IRI_multimodal_project/'

CTRLrna = sc.read_h5ad(datadir+'Stereo-seq_IRI/CTRL_scanpy_scTransform_v2_SD_rna.h5ad')
IRIrna = sc.read_h5ad(datadir+'Stereo-seq_IRI/IRI_scanpy_scTransform_v2_SD_rna.h5ad')

In [31]:
def segmentation(N):
    
    # load image
    kidneyn = Image.open(datadir+'Figures/IRI_regist_K'+N+'_MASK.tif')
    
    # transform image to numpy array
    arr = asarray(kidneyn)
    
    # transform to pandas df
    df = pd.DataFrame(arr)

    filt_df = df.loc[~(df==0).all(axis=1)]
    filt_df = filt_df.loc[:, (filt_df != 0).any(axis=0)]
    
    # subset of the anndata.obs with x and y coordinate and barcode
    df_allk = IRIrna.obs[['x','y']]
    df_allk['barcode'] = IRIrna.obs.index 
    
    # subset the df based on the filtered IMAGE pixels : square around the kidney 
    df_allk = df_allk[df_allk['y'] > filt_df.index[0]]
    df_allk = df_allk[df_allk['y'] < filt_df.index[-1]]
    df_allk = df_allk[df_allk['x'] < filt_df.columns[-1]]
    df_allk = df_allk[df_allk['x'] > filt_df.columns[0]]
    
    # pivot the dataframe so that the y column becomes the index, x the columns and the barcodes the values 
    df_barcodes = df_allk.pivot(index='y', columns='x', values='barcode')
    
    # filter the image df to keep only pixels that are present in the anndata obj
    filt_df = filt_df[filt_df.columns.intersection(list(df_barcodes.columns))]
    filt_df = filt_df.loc[filt_df.index.isin(list(df_barcodes.index))]
    
    # obtain a list of coordinates [row,col] when the value is equal to zero 
    l = [[row, col] for row in filt_df.index for col in filt_df.columns if filt_df.loc[row, col] == 0]
    
    # filter the barcoded df based on the list l
    for i in range(len(l)):
        df_barcodes.loc[l[i][0],l[i][1]] = 0
    
    barcodes = [df_barcodes.loc[row, col] for row in df_barcodes.index for col in df_barcodes.columns if df_barcodes.loc[row, col] != 0]
    barcodes = [x for x in barcodes if str(x) != 'nan']
    
    IRI_samples_kidneyn = IRIrna[IRIrna.obs.index.isin(barcodes),:]

    # save subset anndata
    IRI_samples_kidneyn.write_h5ad(datadir+'Stereo-seq_IRI/bIRI_Kidney'+N+'.h5ad')
    return IRI_samples_kidneyn

#### Kidney 1 

In [32]:
bIRI_kidney1 = segmentation('1')
bIRI_kidney1

/tmp/ipykernel_2872360/99005040.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_allk['barcode'] = IRIrna.obs.index


View of AnnData object with n_obs × n_vars = 48347 × 3000
    obs: 'nCount_RNA', 'nFeature_RNA', 'orig.ident', 'x', 'y', 'nFeautre_RNA', 'percent.mt', 'nCount_SCT', 'nFeature_SCT', 'Seurat_Clusters_Res1', 'Seurat_Clusters_Res1_labelled', 'Seurat_Clusters_Res1_labelled_V2', 'Seurat_Clusters_Res1_labelled_V3'
    var: 'features'
    uns: 'neighbors'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances'

#### Kidney 2

In [33]:
bIRI_kidney2 = segmentation('2')
bIRI_kidney2

/tmp/ipykernel_2872360/99005040.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_allk['barcode'] = IRIrna.obs.index


View of AnnData object with n_obs × n_vars = 40918 × 3000
    obs: 'nCount_RNA', 'nFeature_RNA', 'orig.ident', 'x', 'y', 'nFeautre_RNA', 'percent.mt', 'nCount_SCT', 'nFeature_SCT', 'Seurat_Clusters_Res1', 'Seurat_Clusters_Res1_labelled', 'Seurat_Clusters_Res1_labelled_V2', 'Seurat_Clusters_Res1_labelled_V3'
    var: 'features'
    uns: 'neighbors'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances'

#### Kidney 3

In [34]:
bIRI_kidney3 = segmentation('3')
bIRI_kidney3

/tmp/ipykernel_2872360/99005040.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_allk['barcode'] = IRIrna.obs.index


View of AnnData object with n_obs × n_vars = 40494 × 3000
    obs: 'nCount_RNA', 'nFeature_RNA', 'orig.ident', 'x', 'y', 'nFeautre_RNA', 'percent.mt', 'nCount_SCT', 'nFeature_SCT', 'Seurat_Clusters_Res1', 'Seurat_Clusters_Res1_labelled', 'Seurat_Clusters_Res1_labelled_V2', 'Seurat_Clusters_Res1_labelled_V3'
    var: 'features'
    uns: 'neighbors'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances'

Some quality checks

In [35]:
bIRI_kidney1.shape[0] + bIRI_kidney2.shape[0] + bIRI_kidney3.shape[0]

129759